# 07 – Ytelse per gruppe: rettferdighet og subgrupper
*Data Mining & Analytics – Samling 2, dag 2*

En modell kan ha god ytelse "over all" og likevel fungere dårlig for ulike undergrupper. Dette bør undersøker og henger av og til sammen med etikk og algoritmisk skjevhet.
Vi bruker Titanic og ser på ytelsen for kvinner og menn – og bruker statistikk fra dag 1 for å avgjøre om forskjonnonnellene er reelle.

In [1]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
rng = np.random.default_rng(42)

In [2]:
from sklearn.metrics import confusion_matrix

def klassifiserings_metrikker(y_sann, y_pred):
    """Metrikker for binær klassifisering (positiv klasse = 1)."""
    tn, fp, fn, tp = confusion_matrix(y_sann, y_pred, labels=[0, 1]).ravel()
    return {"nøyaktighet": (tp + tn) / (tp + tn + fp + fn),
            "sensitivitet": tp / (tp + fn) if tp + fn else np.nan,
            "spesifisitet": tn / (tn + fp) if tn + fp else np.nan,
            "presisjon": tp / (tp + fp) if tp + fp else np.nan,
            "F1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan}

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

titanic = pd.read_csv("../Data/titanic.csv")
X = titanic[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]; y = titanic["survived"]
num = ["age", "sibsp", "parch", "fare"]; kat = ["pclass", "sex", "embarked"]
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("kat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), kat)])
modell = Pipeline([("prep", prep), ("m", RandomForestClassifier(300, min_samples_leaf=3, random_state=0))])

# out-of-fold-prediksjoner for ALLE passasjerer (hver prediksjon gjort av en modell som ikke har sett passasjeren)
p = cross_val_predict(modell, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
pred = (p >= 0.5).astype(int)
print("Totalt:", {k: round(v, 3) for k, v in klassifiserings_metrikker(y, pred).items()})

Totalt: {'nøyaktighet': np.float64(0.822), 'sensitivitet': np.float64(0.699), 'spesifisitet': np.float64(0.898), 'presisjon': np.float64(0.81), 'F1': np.float64(0.75)}


## Metrikker per gruppe

In [5]:
rader = {}
for kjonn, idx in titanic.groupby("sex").groups.items():
    m = klassifiserings_metrikker(y.loc[idx], pred[idx])
    m["n"] = len(idx)
    m["andel overlevde (base rate)"] = y.loc[idx].mean()
    m["andel predikert overlevde"] = pred[idx].mean()
    rader[kjonn] = m
print(pd.DataFrame(rader).T.round(3))

        nøyaktighet  sensitivitet  spesifisitet  presisjon     F1      n  \
female        0.806         0.901         0.531      0.847  0.873  314.0   
male          0.830         0.266         0.962      0.617  0.372  577.0   

        andel overlevde (base rate)  andel predikert overlevde  
female                        0.742                      0.790  
male                          0.189                      0.081  


Total nøyaktighet skjuler at sensitiviten er mye lavere for menn, mensspesifisitet er lavere for kvinner.

Forskjellen henger sammen med ulik antall kvinner og menn blant overlevende og modellen lærer at kjønn er en sterk prediktor for overlevelse.

## Er forskjellen i sensitivitet statistisk reell? Bootstrap-KI for differansen

In [6]:
kjonn = titanic["sex"].to_numpy()
y_np = y.to_numpy()

def sens(mask):
    pos = mask & (y_np == 1)
    return (pred[pos] == 1).mean()

obs = sens(kjonn == "female") - sens(kjonn == "male")
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y_np), len(y_np))
    k, y_, p_ = kjonn[i], y_np[i], pred[i]
    s = lambda g: (p_[(k == g) & (y_ == 1)] == 1).mean()
    boot.append(s("female") - s("male"))
print(f"Sensitivitet kvinner − menn = {obs:+.3f},  95 % bootstrap-KI = {np.round(np.percentile(boot, [2.5, 97.5]), 3)}")

# Chi-kvadrat på oversett (FN) blant de som faktisk overlevde
ov = titanic[(y == 1)].assign(riktig=(pred[y == 1] == 1))
krysstab = pd.crosstab(ov["sex"], ov["riktig"])
print("\n", krysstab)
chi2, pv, _, _ = stats.chi2_contingency(krysstab)
print(f"χ²-test: p = {pv:.2g}")

Sensitivitet kvinner − menn = +0.635,  95 % bootstrap-KI = [0.543 0.724]

 riktig  False  True 
sex                 
female     23    210
male       80     29
χ²-test: p = 3.6e-32


## Hva kan vi gjøre?
Ingen fasit, men man kan f.eks lage ulike prediksjonsterskler for undergruppene

In [7]:

m_ = kjonn == "male"; f_ = kjonn == "female"
def sens_t(mask, t):
    pos = mask & (y_np == 1); return (p[pos] >= t).mean()

def spes_t(mask, t):
    neg = mask & (y_np == 0); return (p[neg] < t).mean()

t_menn = min(np.linspace(0.05, 0.95, 91), key=lambda t: abs(sens_t(m_, t) - sens_t(f_, 0.5)))

print(f"Terskel for kvinner 0.50: sens = {sens_t(f_, 0.5):.2f}, spes = {spes_t(f_, 0.5):.2f}")
print(f"Terskel for menn {t_menn:.2f}:     sens = {sens_t(m_, t_menn):.2f}, spes = {spes_t(m_, t_menn):.2f}   (lik sensitivitet, men lavere spesifisitet for menn)")

Terskel for kvinner 0.50: sens = 0.90, spes = 0.53
Terskel for menn 0.05:     sens = 0.91, spes = 0.14   (lik sensitivitet, men lavere spesifisitet for menn)
